In [36]:
import struct
import time

# rtcl_d3xx のインポート
try:
    import rtcl_d3xx
except ImportError:
    # 未インストールの場合はリポジトリからインストールします。
    from pathlib import Path
    import subprocess
    import sys
    package_dir = next(
        (
            root / "python" / "rtcl-d3xx"
            for root in (Path.cwd(), *Path.cwd().resolve().parents)
            if (root / "python" / "rtcl-d3xx" / "pyproject.toml").is_file()
        ),
        None,
    )
    if package_dir is None:
        raise FileNotFoundError(
            "python/rtcl-d3xx が見つかりません。リポジトリ内で実行してください。"
        )

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", str(package_dir)]
    )
    import rtcl_d3xx

In [37]:
# デバイスを初期化
dev = rtcl_d3xx.Fifo32(dev_index=0)

In [38]:
# レジスタ定義
ADDR_ID = 0x0000_0000
ADDR_VERSION = 0x0000_0004
ADDR_USER0 = 0x0000_0008
ADDR_USER1 = 0x0000_000C
ADDR_PUSH_SW = 0x0000_0010
ADDR_DIP_SW = 0x0000_0014
ADDR_LED = 0x0000_0018
ADDR_PMOD = 0x0000_001C


In [39]:

# AXI4-Lite レジスタ読み出し
for name, address in (
    ("ID", ADDR_ID),
    ("VERSION", ADDR_VERSION),
    ("PUSH_SW", ADDR_PUSH_SW),
    ("DIP_SW", ADDR_DIP_SW),
    ("USER0", ADDR_USER0),
    ("USER1", ADDR_USER1),
):
    print(f"read  {name:<7}: 0x{dev.read_axi4l(address):08x}")

# User レジスタ読み書き
print("write USER0   : wdata = 0x12345678 wstrb=0b1111")
dev.write_axi4l(ADDR_USER0, 0x12345678, strb=0b1111)
print("write USER1   : wdata = 0xfedcba98 wstrb=0b1111")
dev.write_axi4l(ADDR_USER1, 0xFEDCBA98, strb=0b1111)
print(f"read  USER0   : 0x{dev.read_axi4l(ADDR_USER0):08x}")
print(f"read  USER1   : 0x{dev.read_axi4l(ADDR_USER1):08x}")

print("write USER0   : wdata = 0xaa55aa55 wstrb=0b1010")
dev.write_axi4l(ADDR_USER0, 0xAA55AA55, strb=0b1010)
print("write USER1   : wdata = 0xaa55aa55 wstrb=0b0101")
dev.write_axi4l(ADDR_USER1, 0xAA55AA55, strb=0b0101)
print(f"read  USER0   : 0x{dev.read_axi4l(ADDR_USER0):08x}")
print(f"read  USER1   : 0x{dev.read_axi4l(ADDR_USER1):08x}")

read  ID     : 0x1234abcd
read  VERSION: 0x00010000
read  PUSH_SW: 0x00000000
read  DIP_SW : 0x00000003
read  USER0  : 0xaa34aa78
read  USER1  : 0xfe55ba55
write USER0   : wdata = 0x12345678 wstrb=0b1111
write USER1   : wdata = 0xfedcba98 wstrb=0b1111
read  USER0   : 0x12345678
read  USER1   : 0xfedcba98
write USER0   : wdata = 0xaa55aa55 wstrb=0b1010
write USER1   : wdata = 0xaa55aa55 wstrb=0b0101
read  USER0   : 0xaa34aa78
read  USER1   : 0xfe55ba55


In [40]:
# LED / PMOD 点滅
for _ in range(3):
    print("LED ON")
    dev.write_axi4l(ADDR_LED, 0x03)
    dev.write_axi4l(ADDR_PMOD, 0xFF)
    time.sleep(0.5)
    print("LED OFF")
    dev.write_axi4l(ADDR_LED, 0x00)
    dev.write_axi4l(ADDR_PMOD, 0x00)
    time.sleep(0.5)

LED ON
LED OFF
LED ON
LED OFF
LED ON
LED OFF


In [41]:

# AXI4-Stream データ送受信
input_data = list(range(1, 11))
print(f"input data: {input_data}")
input_bytes = struct.pack("<10I", *input_data)
dev.send_axi4s(input_bytes, tuser=0)

packet = dev.recv_axi4s(timeout=1.0)
if len(packet.data) != len(input_bytes):
    raise RuntimeError(
        f"AXI4-Stream receive size mismatch: {len(packet.data)} != {len(input_bytes)}"
    )
result = list(struct.unpack("<10I", packet.data))
print(f"result: {result}")


input data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
result: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [42]:
# デバイスを削除してクローズ
del dev

Device closed
